In [215]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import torch
import matplotlib.pyplot as plt
from src.data_processing.utils import set_seed
import pandas as pd
import joblib

from src.models.TimeGAN_torch.timegan_pytorch import TimeGAN
from src.models.TimeGAN_torch.metrics.discriminative_metrics import discriminative_score_metrics
from src.models.TimeGAN_torch.metrics.predictive_metrics import predictive_score_metrics
from src.models.TimeGAN_torch.metrics.visualization_metrics import visualization

from src.data_processing.utils import TimeSeriesDataset, KFoldTimeSeries, save_synth_data

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Need to set random seed for libraries, GPU, etc. for reproducibility 

set_seed(42)

## Set Network Parameters

seq_len is the length of the time series sequences. In this case seq_len=24 which represents 24 consecutive trading days
n_seq is the number of features, the CBOE VIX_history.csv dataset provided 4 features (CLOSE, LOW, HIGH, OPEN)

In [217]:
# Newtork parameters
parameters = dict()

parameters['module'] = 'gru'
parameters['hidden_dim'] = 32
parameters['num_layers'] = 3
parameters['batch_size'] = 128
parameters['seq_len'] = 24
parameters['lr'] = 2e-4
parameters['device'] = torch.device("cuda" if torch.cuda.is_available() else "cpu")
parameters['epochs'] = 250
parameters['dataset_name'] = 'VIX'

In [218]:
data_path = './data/real_data/VIX_History.csv'
scaler_path= f'./model_checkpoints/scalers/scaler_{dataset_name}.pkl'

# Load and preprocess the data
# VIX_history had a repeat values that seemed like garbage, just removed those
all_data = pd.read_csv(data_path).iloc[505:].reset_index(drop=True)

# Remove the DATE column
all_data.drop(['DATE'], axis=1, inplace=True)

parameters['n_seq'] = all_data.shape[1]

# Create dataset object
dataset = TimeSeriesDataset(data=all_data, seq_len=parameters['seq_len'], n_seq=parameters['n_seq'], scaler_path=scaler_path)

# Get train, val, test datasets
train_dataset, val_dataset, test_dataset = dataset.get_datasets()

Initializing TimeSeriesDataset...
Data split: 6688 train, 836 validation, 836 test
Scaler saved at ./model_checkpoints/scalers/scaler_VIX.pkl


In [219]:
# `data` is a numpy array of shape (n_samples, n_features)
# kfolds = KFoldTimeSeries(all_data, seq_len=parameters['seq_len'], n_splits=5)

# Example: Getting datasets for the first fold
# train_dataset, val_dataset = kfolds.get_fold(0)

# Now `train_dataset` and `val_dataset` can be used in training and validation phases.

## Run TimeGAN for synthetic time-series data generation

In [220]:
# Run TimeGAN
model = TimeGAN(parameters)

Model initialized with ID: VIX_20250227_012534_db74712f_n_layers3_seq_len24_n_seq4_hidden_dim32


In [ ]:
model.train(train_dataset, 
            val_dataset, 
            ae_iters=parameters['epochs'], 
            sup_iters=parameters['epochs'], 
            joint_iters=parameters['epochs'], 
            patience = 10)

[Training] Phase 1: Autoencoder
[Autoencoder] Epoch 0/250, Train Loss: 3.1502, Val Loss: 2.1852
[Autoencoder] Epoch 10/250, Train Loss: 0.6039, Val Loss: 0.6648
[Autoencoder] Epoch 20/250, Train Loss: 0.3561, Val Loss: 0.3849
[Autoencoder] Epoch 30/250, Train Loss: 0.2790, Val Loss: 0.2805
[Autoencoder] Epoch 40/250, Train Loss: 0.2343, Val Loss: 0.2482
[Autoencoder] Epoch 50/250, Train Loss: 0.2134, Val Loss: 0.2092
[Autoencoder] Epoch 60/250, Train Loss: 0.1991, Val Loss: 0.2044
[Autoencoder] Epoch 70/250, Train Loss: 0.1907, Val Loss: 0.1949
[Autoencoder] Epoch 80/250, Train Loss: 0.1823, Val Loss: 0.1834
[Autoencoder] Epoch 90/250, Train Loss: 0.1750, Val Loss: 0.1770
[Autoencoder] Epoch 100/250, Train Loss: 0.1685, Val Loss: 0.1680
[Autoencoder] Early stopping at epoch 108 with best Val Loss: 0.1664
[Autoencoder] Restored best model from early stopping.
[Training] Phase 2: Supervisor
[Supervisor] Epoch 0/250, Train Loss: 0.6225, Val Loss: 0.5834
[Supervisor] Epoch 10/250, Train Lo

In [ ]:
model.load_model('./model_checkpoints/VIX/VIX_20250227_010858_88eef6c4_n_layers3_seq_len5_n_seq4_hidden_dim32')

num_synthetic_samples = len(val_dataset) + parameters['seq_len'] - 1
synthetic_data = model.generate(num_samples=num_synthetic_samples)
print("synthetic_data shape:", synthetic_data.shape)

## Evaluate the generated data

In [ ]:
visualization(test_dataset, synthetic_data, 'pca')
visualization(test_dataset, synthetic_data, 'tsne')

## Inverse-transform generated sequences

The generated sequences are normalized using MinMax(), need to inverse transform the data

In [ ]:
# Load the scaler
scaler_path = f'./model_checkpoints/scalers/scaler_{dataset_name}.pkl'
scaler = joblib.load(scaler_path)

# Reshape synthetic_data from (n_samples, n_timesteps, n_features) to (n_samples * n_timesteps, n_features)
n_samples, n_timesteps, n_features = synthetic_data.shape
synthetic_data_reshaped = synthetic_data.reshape(-1, n_features)

# Inverse transform
synthetic_data_inv = scaler.inverse_transform(synthetic_data_reshaped)

# Reshape back to (n_samples, n_timesteps, n_features)
# Each of the samples would represent a single block of time (e.g. synthetic_data_inv[i])
synthetic_data_inv = synthetic_data_inv.reshape(n_samples, n_timesteps, n_features)

# Example of a single block of 24 days
synth_df = pd.DataFrame(synthetic_data_inv[0], columns=all_data.columns)

In [ ]:
save_synth_data(synthetic_data_inv, 
                save_dir='./data/synthetic_data',
                dataset_name=parameters['dataset_name'],
                col_names=all_data.columns)